In [ ]:
#@title Estilo de la clase (ejecutar, no hace falta leer) {display-mode: "form"}
from IPython.display import HTML, display
display(HTML(r'''
<style>
@import url('https://fonts.googleapis.com/css2?family=Work+Sans:wght@400;600&family=Amiri:wght@400;700&display=swap');
.rendered_html, .markdown, .cell .text_cell_render { font-family:'Work Sans',system-ui,sans-serif; color:#122535; }
.rendered_html h1,.rendered_html h2,.rendered_html h3 { font-family:'Amiri',Georgia,serif; color:#00529B; }
.rendered_html h2 { border-bottom:2px solid #00529B; padding-bottom:.2em; }
.rendered_html a { color:#00529B; }
.rendered_html table th { background:#00529B; color:#fff; }
.rendered_html h1,.rendered_html h2,.rendered_html h3 { scroll-margin-top:16px; }
</style>
'''))

# Clase 5 · Regresión lineal y regularización

**Analítica de Datos** · Maestría en Ciencias del Comportamiento · Universidad de San Andrés

**Primavera 2026 · 05/09/2026**

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tomdamelio/analitica_de_datos_alumnos/blob/main/clases/clase-05/notebooks/clase05_python.ipynb)

---

La pregunta de hoy es **¿cuánto bienestar laboral podemos predecir a partir del salario?**, y de ahí
en adelante, **¿cuánto mejora si usamos más variables?**

La idea que queremos ver:

> **El modelo que mejor ajusta los datos que ya viste no es el que mejor predice los que vienen.**
> En la Clase 4 esa perilla era K, la cantidad de vecinos. Hoy va a ser la cantidad de variables
> que le metemos al modelo.

El recorrido tiene seis pasos:

| # | Paso | Qué hacemos |
|---|---|---|
| 1 | Armar la tabla | unir `nimbus_clima` con `nimbus_salario`, como en la Clase 2 |
| 2 | Una recta a ojo | mover β₀ y β₁ a mano y mirar los residuos |
| 3 | Mínimos cuadrados | programar la fórmula, sin sklearn |
| 4 | Medir el ajuste | RSE y R², a mano y después con sklearn |
| 5 | Varias variables | ir agregando predictores y ver qué pasa |
| 6 | El número honesto | partir en entrenamiento y testeo, y volver a medir |

> **Cómo se usa esta notebook.** Las celdas de código se corren con `Shift+Enter`, de arriba
> hacia abajo. Si te salteás una, las de abajo pueden fallar porque dependen de variables
> definidas antes. Las celdas marcadas como **ejercicio** tienen huecos para completar.

> **Ojo con los dos "bienestar".** El de hoy es `bienestar_laboral`, un índice de 0 a 100 de la
> encuesta de clima 2026. **No** es el `bienestar` diario del piloto de la fruta (escala 1 a 7) que
> usaron en las clases 1, 2 y 3. Son dos variables distintas.

## 1. Armar la tabla

Dos tablas. `clima` tiene la encuesta de este año: una fila por empleado, con el índice de
bienestar y 19 variables más. `salario` es el panel de sueldos, con una fila por empleado **y por
año**, así que primero hay que quedarse con un año solo.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Todo sale del espejo público de la materia. Si alguna vez cambia de lugar,
# se toca esta única línea.
BASE = "https://raw.githubusercontent.com/tomdamelio/analitica_de_datos_alumnos/main/data/toy-nimbus/"

clima = pd.read_csv(BASE + "nimbus_clima.csv")
salario = pd.read_csv(BASE + "nimbus_salario.csv")

print("clima  ", clima.shape)
print("salario", salario.shape)
clima.head(3)

`salario` tiene 1.800 filas para 600 empleados: tres años por persona. Nos quedamos con **2025**,
que es el sueldo que tenían cuando contestaron la encuesta.

### ✏️ Consigna 1

Uní las dos tablas para tener, en una sola fila por empleado, su salario y su bienestar.

**a)** `salario` tiene tres años por persona. Quedate con **2025**, que es el sueldo que tenían
cuando contestaron la encuesta.

**b)** Unilas por la columna que comparten. Las dos tienen `empleado_id`.

In [ ]:
# TODO: completá el año que nos interesa
salario_2025 = salario[salario["anio"] == ___][["empleado_id", "salario_mensual"]]
# TODO: completá la columna por la que se unen las dos tablas
datos = clima.merge(salario_2025, on="___")

# El salario en millones deja los números en una escala cómoda de leer.
datos["salario"] = datos["salario_mensual"] / 1_000_000

print(datos.shape)
datos[["empleado_id", "salario", "bienestar_laboral"]].head(3)


Antes de modelar nada, mirá las dos variables que vamos a usar. `salario` está en millones de pesos
y `bienestar_laboral` es un índice de 0 a 100.

In [ ]:
resumen = datos[["salario", "bienestar_laboral"]].describe().round(2)
correlacion = datos["salario"].corr(datos["bienestar_laboral"])

print(f"correlación salario / bienestar: {correlacion:.3f}")
resumen


### El punto de partida: el scatter

Antes de ajustar nada, mirá la nube. Cada punto es una persona: su sueldo en el eje horizontal, lo
que contestó en la encuesta de clima en el vertical.

Dos cosas para registrar ahora, porque vuelven más adelante. La primera es que la nube **sube**:
no es una pelota redonda, se estira en diagonal. Eso es lo que va a poder capturar una recta. La
segunda es que es **ancha**: para un mismo sueldo hay gente con 40 y gente con 80 de bienestar. Esa
dispersión es lo que ningún modelo con una sola variable va a poder explicar, y es la razón por la
que el R² que obtengamos va a ser un tercio y no 0,9.

Todo lo que sigue es una manera de resumir esta nube en una regla.


In [ ]:
# Guardamos las dos variables sueltas: se usan en casi todas las celdas de acá en adelante.
x = datos["salario"].to_numpy()
y = datos["bienestar_laboral"].to_numpy()

def dibujar_nube(ax):
    """Dibuja la nube de puntos con los ejes ya nombrados.

    Se usa en varias figuras, así que vive en una función para no repetirla.
    """
    ax.scatter(x, y, s=14, alpha=0.35, color="#33404a")
    ax.set_xlabel("Salario mensual (millones de $)")
    ax.set_ylabel("Bienestar laboral (0-100)")
    ax.set_ylim(0, 100)

fig, ax = plt.subplots(figsize=(7, 4.5))
dibujar_nube(ax)
ax.set_title("Nimbus: 600 empleados")
plt.tight_layout()
plt.show()

## 2. Una recta a ojo

Una recta son dos números: la ordenada al origen β₀ y la pendiente β₁.

$$\hat{y} = \beta_0 + \beta_1 x$$

donde
- $\hat{y}$ es el bienestar que la recta **predice**,
- $x$ es el salario de esa persona,
- $\beta_0$ es dónde la recta cruza el eje vertical,
- $\beta_1$ es cuánto sube el bienestar por cada millón más de salario.

Movete los sliders hasta que la recta te parezca razonable. Anotá los dos valores: los vamos a
comparar contra la solución exacta.

In [ ]:
from ipywidgets import interact, FloatSlider

# El salario va de 0,9 a 1,6 millones: x = 0 está MUY lejos del panel. Si los
# sliders movieran b0 (la ordenada al origen) y b1 por separado, subir b1 un
# punto desplazaría toda la recta ~1,26 y la inclinaría apenas ~0,7 de punta a
# punta del gráfico: se vería como un desplazamiento, no como un giro.
#
# Por eso los sliders no controlan (b0, b1) directo: controlan (altura, b1),
# donde "altura" es el valor de la recta en el salario PROMEDIO. b0 se
# despeja de ahí. Así la pendiente gira la recta alrededor del centro de la
# nube, que es lo que uno espera de un control que se llama "pendiente".
media_x = x.mean()

def recta_a_ojo(altura=65.0, b1=45.0):
    """Dibuja la recta que definen altura y b1 sobre la nube, con sus residuos."""
    b0 = altura - b1 * media_x
    fig, ax = plt.subplots(figsize=(7, 4.5))
    dibujar_nube(ax)
    grilla = np.linspace(x.min(), x.max(), 100)
    ax.plot(grilla, b0 + b1 * grilla, color="#00529B", lw=2.5)
    ax.set_title(f"bienestar = {b0:.1f} + {b1:.1f} · salario")
    plt.tight_layout()
    plt.show()

interact(
    recta_a_ojo,
    altura=FloatSlider(value=65, min=0, max=100, step=1, description="altura"),
    b1=FloatSlider(value=45, min=-40, max=140, step=1, description="β₁"),
);

### Los residuos

El **residuo** de una persona es la diferencia entre su bienestar real y el que la recta le predice:

$$e_i = y_i - \hat{y}_i$$

donde
- $y_i$ es el bienestar observado de la persona $i$,
- $\hat{y}_i$ es el que predice la recta para su salario.

Y el **RSS** es la suma de todos esos residuos al cuadrado. Es un solo número que dice qué tan mala
es una recta: cuanto más chico, mejor.

$$\text{RSS} = \sum_{i=1}^{n} (y_i - \hat{y}_i)^2$$

### ✏️ Consigna 2

Programá el RSS. Son dos pasos: calcular el residuo de cada persona (lo que la recta le erra) y
después sumar todos esos residuos **al cuadrado**.

El cuadrado no es un capricho: si los sumaras tal cual, los positivos y los negativos se cancelan y
una recta espantosa podría dar cero.

In [ ]:
def rss(b0, b1):
    """Suma de los residuos al cuadrado de la recta b0 + b1*x."""
    # TODO: el residuo es el valor real menos el que predice la recta b0 + b1*x
    residuos = y - (___ + ___ * x)
    # TODO: ¿a qué potencia hay que elevarlos antes de sumar?
    return float((residuos ** ___).sum())

# Una recta deliberadamente mala, para tener contra qué comparar.
rss_mala = rss(25.0, 26.0)
print(f"RSS de la recta (25,0 · 26,0): {rss_mala:,.0f}")


Ahora el mismo widget, pero con dos agregados: los residuos dibujados como segmentos rojos, y el
RSS calculado en vivo en el título.

**Tratá de bajar el RSS lo más que puedas moviendo los sliders**, y anotá el mejor valor que
consigas. Vas a notar dos cosas. Que se puede mejorar bastante rápido al principio, y que después
te trabás: movés un slider y empeora, movés el otro y empeora, y no sabés para dónde seguir. Ese
callejón es exactamente el problema que la próxima sección resuelve de un saque.


In [ ]:
def recta_con_residuos(altura=65.0, b1=45.0):
    """La misma recta, ahora con los residuos dibujados y el RSS en el título.

    Igual que en `recta_a_ojo`: los sliders son (altura, b1), no (b0, b1),
    para que la pendiente gire la recta en vez de correrla.
    """
    b0 = altura - b1 * media_x
    fig, ax = plt.subplots(figsize=(7, 4.5))
    prediccion = b0 + b1 * x
    ax.vlines(x, y, prediccion, color="#B4232E", lw=0.5, alpha=0.5)
    dibujar_nube(ax)
    grilla = np.linspace(x.min(), x.max(), 100)
    ax.plot(grilla, b0 + b1 * grilla, color="#00529B", lw=2.5)
    ax.set_title(f"β₀ = {b0:.1f}   β₁ = {b1:.1f}   RSS = {rss(b0, b1):,.0f}")
    plt.tight_layout()
    plt.show()

interact(
    recta_con_residuos,
    altura=FloatSlider(value=65, min=0, max=100, step=1, description="altura"),
    b1=FloatSlider(value=45, min=-40, max=140, step=1, description="β₁"),
);

## 3. Mínimos cuadrados, sin sklearn

Buscar a ojo el RSS más chico es lento y no tiene garantía. Por suerte, para este problema hay una
fórmula exacta:

$$\hat{\beta}_1 = \frac{\sum_i (x_i - \bar{x})(y_i - \bar{y})}{\sum_i (x_i - \bar{x})^2}
\qquad
\hat{\beta}_0 = \bar{y} - \hat{\beta}_1 \bar{x}$$

donde
- $\bar{x}$ y $\bar{y}$ son los promedios de salario y de bienestar,
- el numerador de $\hat{\beta}_1$ mide si los que están por arriba en salario también lo están en
  bienestar,
- el denominador es cuánto se dispersan los salarios, y es lo que pone la escala.

La segunda fórmula dice algo lindo: la recta pasa **siempre** por el punto de los dos promedios.

### ✏️ Consigna 3

Programá las dos fórmulas de arriba. Fijate que las dos usan los promedios `mx` y `my`, que ya
están calculados en la primera línea.

Más adelante comparamos este resultado contra `sklearn`: si ahí coinciden, programaste bien la
fórmula.


In [ ]:
def minimos_cuadrados(x, y):
    """Devuelve (b0, b1) de la recta que minimiza el RSS.

    Son las ecuaciones (3.4) de James y otros, capítulo 3.
    """
    mx, my = x.mean(), y.mean()
    # TODO: arriba, cuánto se aparta cada uno de SU promedio; abajo, sólo el de x
    b1 = ((x - mx) * (y - ___)).sum() / ((x - ___) ** 2).sum()
    # TODO: la recta pasa por (mx, my). Despejá b0 de my = b0 + b1 * mx
    b0 = my - ___ * mx
    return float(b0), float(b1)

b0_ols, b1_ols = minimos_cuadrados(x, y)
print(f"β₀ = {b0_ols:.2f}")
print(f"β₁ = {b1_ols:.2f}")


Dos comprobaciones antes de seguir, que son la forma de convencerse de que la fórmula hizo lo que
promete y no hay que creerle porque sí.

La primera mueve la recta cinco unidades para cada lado y mira qué pasa con el RSS. Fijate que
moverse **para arriba y para abajo empeora lo mismo**: no es un error de la celda, es la señal de
que estamos justo en el fondo. Si estuviéramos en la ladera, un lado daría mejor que el otro.

La segunda usa la propiedad que mencionamos arriba: la recta de mínimos cuadrados pasa exacto por
el punto formado por los dos promedios.


In [ ]:
# Si nos movemos un poco en cualquier dirección, el RSS empeora.
rss_ols = rss(b0_ols, b1_ols)

for db0, db1 in [(-5, 0), (5, 0), (0, -5), (0, 5)]:
    print(f"  β₀{db0:+3.0f}  β₁{db1:+3.0f}   RSS = {rss(b0_ols + db0, b1_ols + db1):>8,.0f}")
print(f"  la de mínimos cuadrados   RSS = {rss_ols:>8,.0f}   <- la más chica")

# La recta pasa por el punto de los dos promedios.
print(f"\npredicción en el salario promedio: {b0_ols + b1_ols * x.mean():.2f}")
print(f"promedio real de bienestar:        {y.mean():.2f}")


Poné acá abajo los dos valores que habías elegido a ojo. La celda dibuja tu recta contra la de
mínimos cuadrados y compara los dos RSS, así que vas a ver cuánto te faltaba.

Si te acercaste mucho, buena señal: el ojo humano es bastante bueno para esto en dos dimensiones.
El problema es que en la sección 5 vamos a tener veinte variables, y ahí no hay ojo que alcance.


In [ ]:
#@title Tu recta contra la de mínimos cuadrados {display-mode: "form"}
b0_a_ojo = 0.0   #@param {type:"number"}
b1_a_ojo = 45.0  #@param {type:"number"}

fig, ax = plt.subplots(figsize=(7.5, 4.5))
dibujar_nube(ax)
grilla = np.linspace(x.min(), x.max(), 100)
ax.plot(grilla, b0_a_ojo + b1_a_ojo * grilla, color="#C8622A", lw=2.5, ls="--",
        label=f"a ojo: RSS = {rss(b0_a_ojo, b1_a_ojo):,.0f}")
ax.plot(grilla, b0_ols + b1_ols * grilla, color="#00529B", lw=2.5,
        label=f"mínimos cuadrados: RSS = {rss_ols:,.0f}")
ax.legend(loc="lower right")
ax.set_title("La recta elegida a ojo contra la exacta")
plt.tight_layout()
plt.show()

## 4. Medir el ajuste: RSE y R²

El RSS por sí solo no se puede leer, porque depende de cuánta gente haya. Dos medidas lo arreglan.

El **RSE** es el error típico, en las unidades de la respuesta:

$$\text{RSE} = \sqrt{\frac{\text{RSS}}{n - 2}}$$

El **R²** compara contra el peor modelo honesto, que es predecir siempre el promedio. Ese modelo
tiene un RSS propio, que se llama **TSS**:

$$R^2 = 1 - \frac{\text{RSS}}{\text{TSS}}
\qquad \text{con} \qquad
\text{TSS} = \sum_i (y_i - \bar{y})^2$$

donde
- $n$ es la cantidad de personas,
- el $-2$ del RSE es porque estimamos dos parámetros,
- $R^2 = 0$ quiere decir que la recta no le ganó a decir el promedio, y $R^2 = 1$ que pasa exacto
  por todos los puntos.

### ✏️ Consigna 4

Calculá las dos medidas. El TSS ya está hecho: es el RSS del modelo que predice siempre el
promedio.

Ojo con el denominador del RSE: no es `n`, es `n` menos la cantidad de parámetros que estimamos.

In [ ]:
n = len(y)
tss = float(((y - y.mean()) ** 2).sum())

# TODO: ¿cuántos parámetros estimamos en una regresión simple?
rse = np.sqrt(rss_ols / (n - ___))
# TODO: el R² compara el error de nuestra recta contra el del modelo del promedio
r2 = 1 - ___ / tss

print(f"TSS = {tss:,.0f}")
print(f"RSE = {rse:.2f} puntos de bienestar")
print(f"R²  = {r2:.3f}")


Nueve puntos de error sobre un índice que en Nimbus va de 11 a 97, y un tercio de la variación
explicada.

Sobre ese 0,33: en ciencias del comportamiento es un resultado **razonable**, no uno malo. Estamos
prediciendo cómo se siente una persona en su trabajo a partir de un solo número, su sueldo. Las
otras mil cosas que influyen (el jefe, la pareja, cómo durmió, si le gusta lo que hace) no están en
la tabla, y van todas al residuo. Un R² de 0,9 en datos de personas suele ser señal de que algo
está mal medido, no de que el modelo sea bueno.

### Lo mismo con sklearn

Ahora en tres líneas. Lo importante no es que sea más corto, sino verificar que **da exactamente lo
mismo**: sklearn no hace magia, hace la cuenta que acabás de programar.


In [ ]:
from sklearn.linear_model import LinearRegression

X = datos[["salario"]]            # sklearn espera una tabla, no un vector
modelo = LinearRegression().fit(X, y)

print(f"sklearn  b0 = {modelo.intercept_:.4f}   b1 = {modelo.coef_[0]:.4f}   R2 = {modelo.score(X, y):.4f}")
print(f"a mano   b0 = {b0_ols:.4f}   b1 = {b1_ols:.4f}   R2 = {r2:.4f}")


In [ ]:
# Las tres rectas juntas: la que elegiste a ojo, la que programaste y la de sklearn.
# Las dos últimas quedan una encima de la otra, que es justamente el punto.
fig, ax = plt.subplots(figsize=(7.5, 4.5))
dibujar_nube(ax)
grilla = np.linspace(x.min(), x.max(), 100)
ax.plot(grilla, b0_a_ojo + b1_a_ojo * grilla, color="#C8622A", lw=2.5, ls="--",
        label=f"a ojo (RSS = {rss(b0_a_ojo, b1_a_ojo):,.0f})")
ax.plot(grilla, b0_ols + b1_ols * grilla, color="#00529B", lw=6, alpha=0.35,
        label="a mano, con la fórmula")
ax.plot(grilla, modelo.predict(grilla.reshape(-1, 1)), color="#1F7A4D", lw=2,
        label=f"sklearn (R2 = {modelo.score(X, y):.3f})")
ax.legend(loc="lower right")
ax.set_title("La misma recta por dos caminos")
plt.tight_layout()
plt.show()

## 5. Varias variables

La encuesta tiene 19 predictores además del salario. Estos son los candidatos, cada uno solo, con
su R²:

In [ ]:
CANDIDATAS = ["salario", "apoyo_equipo", "reconocimiento", "autonomia",
              "horas_extra_semana", "dias_home_office", "bono_anual_pct",
              "reuniones_semana", "mensajes_chat_dia", "puntualidad_pct",
              "horas_capacitacion", "distancia_oficina_km"]

def r2_de(columnas, tabla=None):
    """R² del modelo que usa esas columnas. Es la función que vas a usar todo el rato."""
    tabla = datos if tabla is None else tabla
    objetivo = tabla["bienestar_laboral"]
    return LinearRegression().fit(tabla[columnas], objetivo).score(tabla[columnas], objetivo)

resultados = {}
for columna in CANDIDATAS:
    resultados[columna] = r2_de([columna])

solas = pd.Series(resultados).sort_values(ascending=False)
solas.round(3)


Hay tres que explican algo y el resto está cerca de cero. La pregunta obvia: si junto las tres
buenas, ¿el R² es la suma de los tres?

### ✏️ Consigna 5

Compará las dos cosas: **sumar** los tres R² por separado contra ajustar **un solo modelo** con las
tres variables juntas.

Anotá tu predicción antes de correr la celda: ¿va a dar más, menos o igual? La respuesta dice algo
sobre cómo se reparte la información entre variables que miden cosas parecidas.


In [ ]:
tres = ["salario", "apoyo_equipo", "reconocimiento"]

suma_individuales = float(solas["salario"] + solas["apoyo_equipo"] + solas["reconocimiento"])
# TODO: ajustá UN modelo con las tres variables a la vez
juntas = r2_de(___)

print(f"sumando los tres R2 por separado:   {suma_individuales:.3f}")
print(f"el modelo con las tres juntas:      {juntas:.3f}")
print(f"diferencia (información repetida):  {suma_individuales - juntas:.3f}")


No se suman: la información se superpone. El que se siente apoyado por su equipo suele ser el mismo
que se siente reconocido, así que ese pedazo se cuenta una sola vez.

### Un coeficiente no se lee solo

El caso más claro del dataset es el bono anual.

In [ ]:
solo_bono = LinearRegression().fit(datos[["bono_anual_pct"]], y)
con_salario = LinearRegression().fit(datos[["bono_anual_pct", "salario"]], y)
r_bono_salario = datos["bono_anual_pct"].corr(datos["salario"])

print(f"beta del bono, solo:            {solo_bono.coef_[0]:+.3f}")
print(f"beta del bono, con el salario:  {con_salario.coef_[0]:+.3f}")
print(f"correlación bono / salario:      {r_bono_salario:.3f}")
print(f"R2 solo bono: {r2_de(['bono_anual_pct']):.3f}   "
      f"con las dos: {r2_de(['bono_anual_pct', 'salario']):.3f}")


El bono parecía un predictor fuerte y resultó ser el salario escrito otra vez: en Nimbus el bono es
un porcentaje del sueldo. Un coeficiente se lee **siempre** en el contexto de qué otras variables
están en el modelo.

### Ahora armá tu modelo

Esta es la parte en la que trabajás vos. Tenés la lista `CANDIDATAS` y la función `r2_de()`.

In [ ]:
# Un ayudante para ir tanteando rápido.
def probar(columnas):
    valor = r2_de(columnas)
    print(f"R2 = {valor:.3f}   con {len(columnas)} variable(s): {', '.join(columnas)}")
    return valor

probar(["salario"])
probar(["salario", "apoyo_equipo"])
probar(["salario", "apoyo_equipo", "reconocimiento"])
probar(["salario", "apoyo_equipo", "reconocimiento", "puntualidad_pct"])
print()
print("Variables disponibles:")
print(", ".join(CANDIDATAS))

Mirá las cuatro pruebas antes de seguir. Agregar `apoyo_equipo` y `reconocimiento` sube el R².
Agregar `puntualidad_pct`, que es ruido puro, lo sube **también**, aunque poquísimo. Guardate eso:
en un rato va a ser el centro de la clase.

### ✏️ Consigna 6

Armá tu propio modelo con **al menos tres variables** de `CANDIDATAS`, usando `probar()` para ir
tanteando. Cuando tengas una combinación que te convenza, ponela en `mis_variables`.

No busques la mejor de todas: buscá una que te parezca razonable. En la próxima sección vamos a ver
qué tan buena era de verdad.

In [ ]:
# TODO: poné acá las variables que elegiste. Al menos tres, todas de CANDIDATAS.
mis_variables = ["___", "___", "___"]

r2_mio = probar(mis_variables)


## 6. El número honesto: entrenamiento y testeo

Todo lo que hiciste hasta acá lo mediste sobre las **mismas 600 personas** con las que armaste el
modelo. Es como corregir un examen con el machete a la vista: el número sale bien, pero no dice si
aprendiste.

Y ya viste el síntoma: en la consigna anterior, agregar `puntualidad_pct` **subió** el R², y esa
variable es ruido. No puede ser que agregar ruido mejore un modelo. Lo que pasa es que el R² medido
sobre los datos con los que ajustaste **nunca baja** cuando agregás una variable: en el peor caso
el modelo le pone un coeficiente casi nulo y queda igual. Nunca peor. Por eso ese número no sirve
para elegir.

La solución es la misma de la Clase 4: apartamos un pedazo de los datos antes de empezar, no lo
tocamos, y recién al final medimos ahí.


### ✏️ Consigna 7

Partí la tabla dejando el **30%** para testeo, y evaluá **tu** modelo en los dos lados.

`random_state=42` es para que a todos les dé lo mismo y podamos comparar en clase.

In [ ]:
from sklearn.model_selection import train_test_split

# TODO: qué proporción va a testeo
entrenamiento, testeo = train_test_split(datos, test_size=___, random_state=42)

mio = LinearRegression().fit(entrenamiento[mis_variables], entrenamiento["bienestar_laboral"])
r2_train_mio = mio.score(entrenamiento[mis_variables], entrenamiento["bienestar_laboral"])
# TODO: el número honesto se mide en el conjunto que el modelo NO vio
r2_test_mio = mio.score(___[mis_variables], ___["bienestar_laboral"])

print(f"tu modelo: {', '.join(mis_variables)}")
print(f"  filas de entrenamiento: {len(entrenamiento)}   filas de testeo: {len(testeo)}")
print(f"  R2 con TODA la data (lo de recién): {r2_mio:.3f}")
print(f"  R2 en entrenamiento:                {r2_train_mio:.3f}")
print(f"  R2 en testeo:                       {r2_test_mio:.3f}")
print(f"  hueco entrenamiento - testeo:       {r2_train_mio - r2_test_mio:+.3f}")


### ¿Se podía elegir mejor?

Comparemos tres modelos sobre la misma partición: sólo el salario, el tuyo, y uno con **todas** las
variables de la tabla.

Mirá las dos columnas por separado. Si agregar variables siempre mejorara, el de **todas** tendría
que ganar en las dos. Prestá atención a la última columna, el hueco entre entrenamiento y testeo:
es la medida de cuánto se está engañando cada modelo a sí mismo.


In [ ]:
# La lista de TODAS las variables de la tabla, y un ayudante que mide de los dos lados.
TODAS = [c for c in datos.columns
         if c not in ["empleado_id", "bienestar_laboral", "salario_mensual"]]

def evaluar(columnas):
    """Ajusta en entrenamiento y devuelve (R2 entrenamiento, R2 testeo)."""
    objetivo = entrenamiento["bienestar_laboral"]
    m = LinearRegression().fit(entrenamiento[columnas], objetivo)
    return (m.score(entrenamiento[columnas], objetivo),
            m.score(testeo[columnas], testeo["bienestar_laboral"]))

print(f"{len(TODAS)} variables disponibles")


Con eso armamos la tabla de los tres modelos. La última columna, el **hueco**, es lo que hay que
mirar: cuánto se está engañando cada modelo a sí mismo.


In [ ]:
modelos = [("sólo el salario", ["salario"]), ("el tuyo", mis_variables), ("TODAS", TODAS)]

filas = []
for nombre, columnas in modelos:
    r2_train, r2_test = evaluar(columnas)
    filas.append({"modelo": nombre, "variables": len(columnas), "train": r2_train,
                  "test": r2_test, "hueco": r2_train - r2_test})

comparacion = pd.DataFrame(filas).set_index("modelo")
comparacion.round(3)


Ahí está la clase entera en una tabla. Con **todas** las variables el R² de entrenamiento es el más
alto de los tres, y el de testeo **no**. El hueco entre las dos columnas es el sobreajuste.

### La curva completa

Para ver dónde estaba el óptimo, armamos una escalera: empezamos con la variable que más explicaba
sola, y le vamos sumando las demás **en el orden en que aparecían en la lista de recién**, de la
que más explicaba a la que menos. En cada escalón medimos de los dos lados.

Es a mano y a propósito. Hay 4.095 combinaciones posibles de estas doce variables, y no hace falta
recorrerlas para ver lo que queremos ver.


In [ ]:
orden = list(solas.index)

historia = []
for k in range(1, len(orden) + 1):
    r2_train, r2_test = evaluar(orden[:k])          # las primeras k de la lista
    historia.append({"n_variables": k, "agregada": orden[k - 1],
                     "r2_entrenamiento": r2_train, "r2_testeo": r2_test})

historia = pd.DataFrame(historia)
mejor = historia.loc[historia["r2_testeo"].idxmax()]

print(f"mejor en testeo: {int(mejor['n_variables'])} variables, R2 = {mejor['r2_testeo']:.3f}")
print(f"tu modelo:       {len(mis_variables)} variables, R2 = {r2_test_mio:.3f}")
historia.round(3)


In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 4.5))
ax.plot(historia["n_variables"], historia["r2_entrenamiento"], "o-",
        color="#1F6FB4", label="entrenamiento")
ax.plot(historia["n_variables"], historia["r2_testeo"], "o-",
        color="#C8622A", label="testeo")
ax.axvline(mejor["n_variables"], color="#33404a", ls="--", lw=1)
ax.scatter([len(mis_variables)], [r2_test_mio], s=160, marker="*",
           color="#1F7A4D", zorder=5, label="tu modelo, en testeo")
ax.set_xlabel("Variables en el modelo")
ax.set_ylabel("R2")
ax.set_title("El entrenamiento nunca baja. El testeo sí.")
ax.legend(loc="lower right", fontsize=9)
plt.tight_layout()
plt.show()


La curva azul sube siempre; la naranja sube, hace un máximo y después se cae. Ese máximo es la
cantidad de variables que conviene usar, y **no es la mayor**. La estrella verde es dónde quedó tu
modelo.

Fijate en el segundo escalón, que es el más interesante de la tabla: agregar `bono_anual_pct` no
mueve **ninguno** de los dos R². Es la misma historia de hace un rato, ahora medida: el bono es el
salario escrito con otro nombre, así que no trae información nueva. Una variable puede tener un R²
alto ella sola y aportar exactamente cero al lado de las demás.

### Lo que queda: penalizar en vez de descartar

Elegir variables a mano funciona, pero es lento, depende de la partición que te tocó, y obliga a
decidir de a una si cada variable entra o sale. Cuando hay pocos datos para muchas variables hay
una alternativa: dejarlas todas adentro y **penalizar** los coeficientes para que el modelo no se
entusiasme con ninguna.


### ✏️ Consigna 8

Comprobá con los datos lo que dijimos en la teoría. Entrená los tres modelos con **sólo 30
personas** y las 19 variables, y comparalos sobre el resto.

`alpha` es el λ de la teoría. Probá con 10 para Ridge y 0.5 para Lasso. El `StandardScaler` del
pipeline es lo que estandariza antes de penalizar.

In [ ]:
from sklearn.linear_model import Lasso, Ridge
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# TODO: cuántas personas usamos para entrenar
chico, resto = train_test_split(datos, train_size=___, random_state=11)

# TODO: alpha es el λ de la teoría. Probá 10 para Ridge y 0.5 para Lasso.
modelos = {"mínimos cuadrados": LinearRegression(),
           "Ridge": Ridge(alpha=___),
           "Lasso": Lasso(alpha=___, max_iter=50_000)}

for nombre, receta in modelos.items():
    m = make_pipeline(StandardScaler(), receta).fit(chico[TODAS], chico["bienestar_laboral"])
    r2_testeo = m.score(resto[TODAS], resto["bienestar_laboral"])
    print(f"{nombre:>18}: R2 de testeo = {r2_testeo:+.3f}")


Con 30 personas y 19 variables, mínimos cuadrados da un R² **negativo**: predice peor que decir el
promedio y no pensar. El lasso, con exactamente los mismos 30, sigue funcionando.

## Hoja de referencia

| Qué | Fórmula | En Python |
|---|---|---|
| Residuo | $e_i = y_i - \hat{y}_i$ | `y - modelo.predict(X)` |
| RSS | $\sum_i e_i^2$ | `((y - pred) ** 2).sum()` |
| RSE | $\sqrt{\text{RSS}/(n-2)}$ | `np.sqrt(rss / (n - 2))` |
| R² | $1 - \text{RSS}/\text{TSS}$ | `modelo.score(X, y)` |
| Ajustar | | `LinearRegression().fit(X, y)` |
| Partir | | `train_test_split(datos, test_size=0.3, random_state=42)` |
| Ridge | RSS $+ \lambda \sum \beta_j^2$ | `Ridge(alpha=10)` |
| Lasso | RSS $+ \lambda \sum \lvert\beta_j\rvert$ | `Lasso(alpha=0.5)` |

> **Antes de regularizar, siempre `StandardScaler`.** Si no, la variable que viene en millones y la
> que viene del 1 al 10 pagan penalizaciones que no son comparables.

## Para el trabajo práctico

Sobre el dataset de tu grupo:

1. Elegí una variable respuesta **numérica**.
2. Ajustá la regresión simple con el predictor más prometedor y reportá el **R² de testeo**.
3. Agregá predictores de a uno y armá la curva de R² de entrenamiento contra R² de testeo, como la
   figura de la sección 6. Marcá dónde se separan.
4. Una línea de conclusión: ¿el mejor modelo es el que más variables tiene? ¿Por qué?